# Tutorial 7: The Logit Lens

**Prerequisites:** T03 (Residual Stream), T06 (First Investigation)

**What you'll learn:**
- How the logit lens reveals what the model "thinks" at each layer
- Why predictions converge (or don't) across layers
- The tuned lens improvement
- How to interpret prediction trajectories

---

## The Core Idea

The **logit lens** (nostalgebraist, 2020) is a beautifully simple idea:

> At every layer, project the residual stream through the unembedding matrix to see what the model would predict **if it stopped here**.

Normally, only the final layer's residual goes through unembedding. But since the unembedding is just a linear projection, we can apply it at *any* layer:

```
early_logits = LayerNorm(residual_at_layer_i) @ W_U
```

This tells us: **how does the model's prediction form, layer by layer?**

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_lens import logit_lens

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 42, 17, 88, 55, 42, 17])
print('Model ready: 4 layers')

## Applying the Logit Lens

In [ ]:
# Run the logit lens
all_logits = logit_lens(model, tokens, apply_ln=True)
print(f'Logit lens output shape: {all_logits.shape}')
print(f'  = [{all_logits.shape[0]} stages, {all_logits.shape[1]} positions, {all_logits.shape[2]} vocab]')
print(f'  Stage 0 = after embedding, Stages 1-4 = after each layer')
print()

# Show what each layer predicts at the last position
print(f'Prediction at last position, by layer:')
stage_names = ['Embed'] + [f'Layer {i}' for i in range(cfg.n_layers)]

for stage in range(all_logits.shape[0]):
    logits_at_pos = np.array(all_logits[stage, -1])
    top_token = np.argmax(logits_at_pos)
    top_logit = logits_at_pos[top_token]
    probs = np.exp(logits_at_pos - logits_at_pos.max())
    probs = probs / probs.sum()
    top_prob = probs[top_token]
    
    print(f'  {stage_names[stage]:8s}: token {top_token:3d} '
          f'(logit={top_logit:+.3f}, prob={top_prob:.3f})')

In [ ]:
# Full logit lens table: every position × every layer
print('Top prediction at each (layer, position):')
print(f'{"":8s}', end='')
for pos in range(len(tokens)):
    print(f'  pos {pos}', end='')
print()

for stage in range(all_logits.shape[0]):
    print(f'{stage_names[stage]:8s}', end='')
    for pos in range(len(tokens)):
        top = int(np.argmax(np.array(all_logits[stage, pos])))
        print(f'  {top:>4d} ', end='')
    print()

print(f'\nStable columns = the prediction was decided early.')
print(f'Changing columns = later layers are still refining the prediction.')

## Prediction Confidence Across Layers

We can also track how **confident** the model is at each layer, not just what it predicts. Confidence typically increases through layers as the model commits to a prediction.

In [ ]:
# Track entropy (uncertainty) at each layer for the last position
print('Prediction entropy by layer (lower = more confident):')

for stage in range(all_logits.shape[0]):
    logits_at_pos = all_logits[stage, -1]
    probs = jax.nn.softmax(logits_at_pos)
    entropy = -float(jnp.sum(probs * jnp.log(probs + 1e-10)))
    max_entropy = float(jnp.log(jnp.array(cfg.d_vocab)))  # Uniform distribution
    bar = '#' * int(entropy / max_entropy * 30)
    print(f'  {stage_names[stage]:8s}: {entropy:.3f} / {max_entropy:.2f}  {bar}')

print(f'\nDecreasing entropy = the model is becoming more confident.')
print(f'Increasing entropy = the model is becoming less certain (unusual).')

## When Do Predictions "Commit"?

An interesting question: **at which layer does the model commit to its final prediction?** We can measure this by checking when the top prediction stops changing.

In [ ]:
from irtk.residual_stream import token_prediction_trajectory

# Track the prediction trajectory at each position
for pos in range(len(tokens)):
    trajectory = token_prediction_trajectory(model, tokens, pos=pos)
    
    # Find when the final prediction first appears
    final_token = trajectory[-1]['top_token']
    commit_layer = None
    for entry in trajectory:
        if entry['top_token'] == final_token and commit_layer is None:
            commit_layer = entry['layer']
    
    preds = [str(e['top_token']) for e in trajectory]
    print(f'Position {pos}: predictions = [{", ".join(preds)}], '
          f'commits at layer {commit_layer}')

## The Tuned Lens

The basic logit lens has a limitation: early layers' residual streams may not be in the right "format" for the unembedding matrix, since the unembedding was trained to work with the final layer's output.

The **tuned lens** (Belrose et al., 2023) fixes this by learning a small affine transformation per layer:

```
tuned_logits_i = (A_i @ LayerNorm(residual_i) + b_i) @ W_U
```

This gives more accurate per-layer predictions, especially for early layers.

In [ ]:
from irtk.logit_lens import TunedLens, train_tuned_lens

# Train a tuned lens on some example data
rng = jax.random.PRNGKey(0)

# Generate training examples
training_tokens = []
for i in range(50):
    rng, subkey = jax.random.split(rng)
    t = jax.random.randint(subkey, (7,), 0, cfg.d_vocab)
    training_tokens.append(t)

# Train the tuned lens
tuned = train_tuned_lens(model, training_tokens, n_epochs=20, lr=1e-3)
print(f'Tuned lens trained: {len(tuned.probes)} layer probes')

# Compare basic vs tuned lens
basic = logit_lens(model, tokens, apply_ln=True)
tuned_result = tuned.apply(model, tokens)

print(f'\nComparison at last position:')
print(f'{"Layer":>8s}  {"Basic top":>10s}  {"Tuned top":>10s}  {"Final":>6s}')
final_pred = int(np.argmax(np.array(basic[-1, -1])))
for stage in range(basic.shape[0]):
    basic_top = int(np.argmax(np.array(basic[stage, -1])))
    tuned_top = int(np.argmax(np.array(tuned_result[stage, -1])))
    b_match = '*' if basic_top == final_pred else ' '
    t_match = '*' if tuned_top == final_pred else ' '
    print(f'{stage_names[stage]:8s}  {basic_top:>5d} {b_match:>4s}  '
          f'{tuned_top:>5d} {t_match:>4s}  {final_pred:>5d}')

print(f'\n* = matches final prediction')

## Practical Uses of the Logit Lens

The logit lens is useful for:

1. **Understanding layer roles**: Which layers refine predictions vs. which change them?
2. **Identifying key layers**: If the prediction flips at layer 3, that layer is worth investigating
3. **Debugging**: If a model gets the wrong answer, when did it go wrong?
4. **Comparing models**: How does prediction formation differ between architectures?

## Key Takeaways

1. **The logit lens** projects intermediate residual streams through the unembedding to see per-layer predictions
2. **Predictions typically converge** through layers — earlier layers are uncertain, later layers commit
3. **The tuned lens** improves early-layer predictions by learning per-layer transformations
4. **Commit depth** tells you how early the model decides on its answer
5. **Prediction flips** between layers indicate where important computation happens

**Next: [T08 — Activation Patching](T08_activation_patching.ipynb)** — The gold standard for testing causal hypotheses about model internals.